# Figure 1(A): Orthogonal Regularization Trade-off

This notebook reproduces Figure 1(A):
- X-axis: Regularization strength lambda
- Y-axis (left): Accuracy (%)
- Y-axis (right): Routing Entropy

**Key Observation**: Strong orthogonal regularization degrades accuracy due to increased routing uncertainty.

In [ ]:
import sys
sys.path.insert(0, '../..')

import json
import os
from pathlib import Path
from collections import defaultdict

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from tqdm.notebook import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## 1. Configuration

Define checkpoints for different lambda values. Update paths to match your checkpoint locations.

In [ ]:
PROJECT_ROOT = '.'
BASE_MODEL_PATH = f'{PROJECT_ROOT}/data/llama-2-7b'
DATASET_DIR = f'{PROJECT_ROOT}/data/flan_v2_subset'

LAMBDA_CONFIGS = {
    0.0: {'checkpoint': f'{PROJECT_ROOT}/output/fig1a_ortho_reg/0_baseline/sft_lora_model', 'accuracy': 37.10},
    0.1: {'checkpoint': f'{PROJECT_ROOT}/output/fig1a_ortho_reg/lambda_0.1/sft_lora_model', 'accuracy': 37.67},
    0.25: {'checkpoint': f'{PROJECT_ROOT}/output/fig1a_ortho_reg/lambda_0.25/sft_lora_model', 'accuracy': 36.98},
    0.5: {'checkpoint': f'{PROJECT_ROOT}/output/fig1a_ortho_reg/lambda_0.5/sft_lora_model', 'accuracy': 37.21},
    1.0: {'checkpoint': f'{PROJECT_ROOT}/output/fig1a_ortho_reg/lambda_1.0/sft_lora_model', 'accuracy': 37.75},
}

print('=' * 60)
print('Checkpoint Path Verification:')
print('=' * 60)
for lam, config in LAMBDA_CONFIGS.items():
    ckpt_path = Path(config['checkpoint'])
    status = 'OK' if ckpt_path.exists() else 'NOT FOUND'
    print(f'lambda={lam}: [{status}] {config["checkpoint"]}')
print('=' * 60)

## 2. Routing Entropy Collector

Hook-based collector for routing weights. Entropy: H = -sum(pi * log(pi))

In [ ]:
class RoutingEntropyCollector:
    def __init__(self):
        self.routing_weights = defaultdict(list)
        self.hooks = []

    def _make_hook(self, layer_name):
        def hook(module, input, output):
            if hasattr(module, 'lora_route') and hasattr(module, 'lora_num'):
                x = input[0]
                x_for_route = x.to(module.lora_route.weight.dtype)
                route_logits = module.lora_route(x_for_route)
                if hasattr(module, 'enable_fine_grained_routing') and module.enable_fine_grained_routing:
                    batch_size = route_logits.shape[0]
                    if len(route_logits.shape) == 3:
                        seq_len = route_logits.shape[1]
                        route_logits = route_logits.view(batch_size, seq_len, module.num_groups, module.lora_num)
                    else:
                        route_logits = route_logits.view(batch_size, module.num_groups, module.lora_num)
                route_weight = F.softmax(route_logits, dim=-1)
                self.routing_weights[layer_name].append(route_weight.detach().cpu())
        return hook

    def register_hooks(self, model):
        from peft.tuners.lora import LoraLinear
        for name, module in model.named_modules():
            if isinstance(module, LoraLinear) and hasattr(module, 'lora_route'):
                hook = module.register_forward_hook(self._make_hook(name))
                self.hooks.append(hook)
        print(f'Registered {len(self.hooks)} routing hooks')
        return self

    def remove_hooks(self):
        for hook in self.hooks:
            hook.remove()
        self.hooks.clear()

    def compute_entropy(self):
        all_entropies = []
        entropy_per_layer = {}
        for layer_name, weight_list in self.routing_weights.items():
            layer_entropies = []
            for route_weight in weight_list:
                eps = 1e-10
                entropy = -torch.sum(route_weight * torch.log(route_weight + eps), dim=-1)
                mean_entropy = entropy.mean().item()
                layer_entropies.append(mean_entropy)
            entropy_per_layer[layer_name] = np.mean(layer_entropies)
            all_entropies.extend(layer_entropies)
        return {'mean': np.mean(all_entropies) if all_entropies else 0.0, 'std': np.std(all_entropies) if all_entropies else 0.0, 'per_layer': entropy_per_layer, 'num_layers': len(entropy_per_layer)}

    def clear(self):
        self.routing_weights.clear()

In [ ]:
def load_dataset_samples(dataset_dir, max_samples=500):
    import random
    for fname in ['train.json', 'train_256/train.json', 'train_512/train.json']:
        train_file = Path(dataset_dir) / fname
        if train_file.exists():
            break
    else:
        raise FileNotFoundError(f'No training file found in {dataset_dir}')
    with open(train_file) as f:
        data = json.load(f)
    if len(data) > max_samples:
        random.seed(42)
        data = random.sample(data, max_samples)
    print(f'Loaded {len(data)} samples from {train_file}')
    return data

samples = load_dataset_samples(DATASET_DIR, max_samples=500)

## 3. Analyze Each Checkpoint

In [ ]:
from transformers import AutoTokenizer, LlamaForCausalLM
from peft import PeftModel
from peft.utils.transformers_patch import patch_llama_for_hydralora

patch_llama_for_hydralora()

def analyze_checkpoint(checkpoint_path, samples, batch_size=4):
    print(f'Loading model from {checkpoint_path}...')
    model = LlamaForCausalLM.from_pretrained(BASE_MODEL_PATH, torch_dtype=torch.bfloat16, device_map={'': device})
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_PATH, use_fast=False, padding_side='left')
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = PeftModel.from_pretrained(model, checkpoint_path)
    model = model.bfloat16()
    model.eval()
    collector = RoutingEntropyCollector()
    collector.register_hooks(model)
    texts = []
    for sample in samples:
        if isinstance(sample, dict):
            text = sample.get('instruction', '') + ' ' + sample.get('input', '') if 'instruction' in sample else sample.get('input', str(sample))
        else:
            text = str(sample)
        texts.append(text)
    print(f'Running inference on {len(texts)} samples...')
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size)):
            batch_texts = texts[i:i+batch_size]
            inputs = tokenizer(batch_texts, return_tensors='pt', padding=True, truncation=True, max_length=512)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            inputs['task_types'] = torch.zeros(len(batch_texts), dtype=torch.long, device=device)
            try:
                _ = model(**inputs)
            except Exception as e:
                print(f'Batch {i} failed: {e}')
                continue
    entropy_stats = collector.compute_entropy()
    collector.remove_hooks()
    del model
    torch.cuda.empty_cache()
    return entropy_stats

In [ ]:
results = {}
for lam, config in LAMBDA_CONFIGS.items():
    ckpt_path = config['checkpoint']
    if not Path(ckpt_path).exists():
        print(f'\nSkipping lambda={lam} (checkpoint not found)')
        continue
    print(f'\n{"=" * 50}')
    print(f'Analyzing lambda={lam}')
    print(f'{"=" * 50}')
    entropy_stats = analyze_checkpoint(ckpt_path, samples)
    results[lam] = {'accuracy': config['accuracy'], 'entropy_mean': entropy_stats['mean'], 'entropy_std': entropy_stats['std'], 'num_layers': entropy_stats['num_layers']}
    print(f'lambda={lam}: Accuracy={config["accuracy"]:.2f}%, Entropy={entropy_stats["mean"]:.4f}')

## 4. Plot Figure 1(A)

In [ ]:
lambdas = sorted(results.keys())
accuracies = [results[lam]['accuracy'] for lam in lambdas]
entropies = [results[lam]['entropy_mean'] for lam in lambdas]

fig, ax1 = plt.subplots(figsize=(8, 6))
color1 = '#1f77b4'
ax1.set_xlabel('Regularization Strength (lambda)', fontsize=12)
ax1.set_ylabel('Accuracy (%)', color=color1, fontsize=12)
line1 = ax1.plot(lambdas, accuracies, 'o-', color=color1, linewidth=2, markersize=8, label='Accuracy')
ax1.tick_params(axis='y', labelcolor=color1)
ax1.set_ylim([min(accuracies) - 2, max(accuracies) + 2])

ax2 = ax1.twinx()
color2 = '#ff7f0e'
ax2.set_ylabel('Routing Entropy', color=color2, fontsize=12)
line2 = ax2.plot(lambdas, entropies, 's--', color=color2, linewidth=2, markersize=8, label='Routing Entropy')
ax2.tick_params(axis='y', labelcolor=color2)

lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper left', fontsize=10)
plt.title('Figure 1(A): Regularization-Routing Trade-off', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fig1a_routing_entropy.png', dpi=300, bbox_inches='tight')
plt.show()
print('\nFigure saved to fig1a_routing_entropy.png')

## 5. Summary Table

In [ ]:
import pandas as pd
df = pd.DataFrame([{'lambda': lam, 'Accuracy (%)': results[lam]['accuracy'], 'Routing Entropy': f"{results[lam]['entropy_mean']:.4f}", 'Entropy Std': f"{results[lam]['entropy_std']:.4f}", 'Num Layers': results[lam]['num_layers']} for lam in sorted(results.keys())])
print('\n' + '=' * 60)
print('Figure 1(A) Results Summary')
print('=' * 60)
print(df.to_string(index=False))
print('=' * 60)